## 📑 Core Modifications & Modifications with Haircut
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/EgBOGQ)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()


from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
import boto3
import os
 
#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ["S3PREFIX"]

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

###### Schema of the table

In [0]:
# Define the schema name
schema_name = f"`{env_prefix}-cdh-delivery`.repoint"

# Create the schema if it does not exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

table_name = 'Core_Modifications' 

schema_core = (
"BusinessDate date,"
"AssetSegment string,"
"ModificationDate date,"
"PartyTaxRegistrationNumber string,"
"CDI string,"
"AgreementKey string,"
"OldExposureKey string,"
"NewExposureKey string,"
"ModificationAmount decimal(19,2),"
"HaircutAmount decimal(19,2),"
"BalancePrvM decimal(19,2),"
"IntermediateAccounts string,"
"ModificationProductCode string,"
"Category string,"
"Categoryfinal string,"
"OperationalBucket string,"
"IFRSGrossCarryingAmountPrvM decimal(19,2),"
"RiskBucketWithMateriality string,"
"NPLIndicatorWithMateriality string,"
"DenouncementIndicator string,"
"EBAStatus string,"
"SPV string,"
"Month string,"
"Category2 string,"
"Categoryfinal2 string,"
"EBAStatusOldExposureFinal int,"
"RestructureType string,"
"PosCourtDecPosHaircutFlag int,"
"NonCoreFlag string,"
"ModificationAmountHaircut decimal(19,2),"
"InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema_core
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### Create Delta Tables with selected fields & Temporary Views for each delta table

##### Dexposure

In [0]:
Dexposure = spark.sql(
f'''
SELECT 
    A.BusinessDate
    ,A.PortfolioName
    ,A.ExposureKey
    , A.OutstandingTotalBalanceMLT  
    , A.Bucket
    ,A.SPV
    ,A.SourceSystem
    ,A.IsActiveFlagInt
    ,A.ProductKey
FROM `{env_prefix}-cdh`.derivative.dexposure as A
where A.BusinessDate = last_day(add_months({run_date!r},-1))
and SourceSystem ='qca'
and PortfolioName like 'Core Retail%' 
and IsActiveFlagInt = 1
and ProductKey NOT IN ('3603','3101', '3246','3453','3444','3409','3475','3443','3604','3497','3483','3492','3484','3592','2143','3520','3494')  
'''
)
Dexposure.createOrReplaceTempView("Dexposure")

##### Dailyportfolio

In [0]:
Dailyportfolio = spark.sql(
f'''
select ACCOUNT_NUMBER,CASE_BUCKET, cast(TOTAL_BALANCE_LCY as decimal(19,2)) as TOTAL_BALANCE_LCY, BusinessDate 
from `{env_prefix}-cdh`.landed.repointqcab_daily_portfolio where BusinessDate = last_day(date_add({run_date!r},-32))
'''
)
Dailyportfolio.createOrReplaceTempView("Dailyportfolio")

##### LPFX364

In [0]:
LPFX364_temp = spark.sql(
f'''
select-- PortfolioName
RestructuringDate
,NewExposureKey
,OldExposureKey
,RestructuringTypeCode
,RestructuringType
,cast(REPLACE(REPLACE(TotalOutstandingAmount, '.', ''), ',', '.')AS DECIMAL(19,2)) as TotalOutstandingAmount
,ExposureType
,RestructuringProductKey
,AgreementKey
,CDI
,PartyTaxRegistrationNumber
,BusinessDate
,IngestionTimeStamp
from `{env_prefix}-cdh`.externalfiles.extlpfxb364
'''
)
LPFX364_temp.createOrReplaceTempView("LPFX364_temp")

##### LPFX302

In [0]:
LPFXB302_temp = spark.sql(
f'''
select  CDI
,DebtorName PartyName
,ExposureKey
,ExposureType
,Product ProductKey
,DateTurningToWrittenOff
,cast(REPLACE(REPLACE(AmountToWrittenOff, '.', ''), ',', '.')AS DECIMAL(19,2)) as AmountToWrittenOff
,cast(REPLACE(REPLACE(AccountingAmounts, '.', ''), ',', '.')AS DECIMAL(19,2)) as AccountingAmounts
,cast(REPLACE(REPLACE(UnaccountedInterest, '.', ''), ',', '.')AS DECIMAL(19,2)) as UnaccountedInterest
,AgreementKey
,RelationshipManagementUnit
,RelatedAccountNumber
,SecuritizationCode
,SecuritizationCodeEOM
,CurrentAccountStatement
,AccountCancellationDate
,BusinessDate
,IngestionTimeStamp
 from `{env_prefix}-cdh`.externalfiles.extlpfxb302
'''
)
LPFXB302_temp.createOrReplaceTempView("LPFXB302_temp")

##### extunionfile

In [0]:
extunionfile_temp = spark.sql(
f'''
select ExposureNumber
, SnapshotDate
,to_date(ingestiontimestamp) as ingestiontimestamp
,cast(replace(IFRSGrossCarryingAmount,',','') as decimal(19,2)) as IFRSGrossCarryingAmount
,BucketCategoryWithMateriality
,NPLWithMaterialityFlag
,DeNuncementFlag
,FPLUnderProbationFlag
,EBAStatus
,NonCoreFlag
from `{env_prefix}-cdh`.externalfiles.extunionfile 
where SnapshotDate = LAST_DAY(ADD_MONTHS({run_date!r}, -1)) 
and IngestionTimeStamp = (Select Max(IngestionTimeStamp) from `{env_prefix}-cdh`.externalfiles.extunionfile  where SnapshotDate = LAST_DAY(ADD_MONTHS({run_date!r}, -1)))
'''
)
extunionfile_temp.createOrReplaceTempView("extunionfile_temp")

previous_month_last_date = spark.sql(f'''SELECT LAST_DAY(ADD_MONTHS({run_date!r}, -1)) ''').collect()[0][0]

# Check if the dataset is empty and print a message accordingly
if extunionfile_temp.count() == 0:
    print(f"The required external file extunionfile dataset for date {previous_month_last_date!r} is empty")

##### Qualco

In [0]:
Qualco = spark.sql(
f'''
SELECT 
    dex.ExposureKey as ExposureKey
    , dp.TOTAL_BALANCE_LCY
    , dp.CASE_BUCKET as Bucket
    ,dex.SPV
FROM Dexposure  as dex
left join DailyPortfolio as dp on dex.ExposureKey = dp.Account_Number and dex.BusinessDate = dp.BusinessDate
where dex.BusinessDate = last_day(date_add({run_date!r},-32))
and SourceSystem ='qca'
and PortfolioName like 'Core Retail%' 
and IsActiveFlagInt = 1
and ProductKey NOT IN ('3603','3101', '3246','3453','3444','3409','3475','3443','3604','3497','3483','3492','3484','3592','2143','3520','3494')  
'''
)
Qualco.createOrReplaceTempView("Qualco")

##### LPFX364

In [0]:
LPFX364 = spark.sql(
f'''
SELECT DISTINCT *,
    RANK() OVER (
        PARTITION BY NewExposureKey, OldExposureKey
        ORDER BY
            BusinessDate DESC,
            IngestionTimeStamp DESC,
            to_date(RestructuringDate, 'dd/MM/yyyy') DESC
    ) AS RNK
FROM LPFX364_temp
WHERE 1=1
AND RestructuringType NOT IN (
    'ΠΕΕ 47 Wholesale',
    'Calypso Proactive Management'
)
AND to_date(RestructuringDate, 'dd/MM/yyyy') <= {run_date!r}
AND to_date(RestructuringDate, 'dd/MM/yyyy') >= date_trunc('MM', {run_date!r})
AND to_date(BusinessDate) <= to_date({run_date!r})
'''
)

LPFX364.createOrReplaceTempView("LPFX364")

##### LPFX302_step0

In [0]:
LPFXB302_step0 = spark.sql(
f'''
SELECT *
	          , RANK() OVER (PARTITION BY ExposureKey
			  ORDER BY BusinessDate DESC, IngestionTimeStamp DESC) AS RNK
	   FROM LPFXB302_temp 

'''
)
LPFXB302_step0.createOrReplaceTempView("LPFXB302_step0")

##### LPFXB302

In [0]:
LPFXB302 = spark.sql(
f'''
SELECT *
	   FROM LPFXB302_step0
	   WHERE RNK=1
'''
)
LPFXB302.createOrReplaceTempView("LPFXB302")

##### Modification1


In [0]:
Modifications1 = spark.sql(
f'''
Select  CASE WHEN L.ExposureType = 'ΕΠΙΧΕΙΡΗΜΑΤΙΚΟ' THEN 'SB'
			      WHEN L.ExposureType = 'ΚΑΤΑΝΑΛΩΤΙΚΟ'   THEN 'CL'
				  WHEN L.ExposureType = 'ΣΤΕΓΑΣΤΙΚΟ'     THEN 'ML' 
				  ELSE NULL END                                         				AS AssetSegment
					,to_date(L.RestructuringDate, 'dd/MM/yyyy')							AS RestructuringDate
					,L.PartyTaxRegistrationNumber										AS PartyTaxRegistrationNumber
					,CONCAT('0', L.CDI)													AS CDI
					, CONCAT('00',L.AgreementKey)                     					AS AgreementKey
					, L.OldExposureKey													AS OldExposureKey 
		  		, L.NewExposureKey														AS NewExposureKey  
				,CASE WHEN SUBSTRING(L.NewExposureKey,4,2)='05' THEN L.TotalOutstandingAmount / COALESCE(E.ExchangeRate,1) ELSE L.TotalOutstandingAmount END AS RestruturingAmount 
				,COALESCE(C.TOTAL_BALANCE_LCY,U.IFRSGrossCarryingAmount)    AS BalancePrvM
				,CASE WHEN COALESCE(H1.ProductKey,H2.ProductKey) = '3410' THEN  COALESCE(H1.ExposureKey,H2.ExposureKey) 
				ELSE NULL END AS IntermediateAccounts
				,L.RestructuringProductKey           			       AS RestructuringProductKey  
				,CASE WHEN L.RestructuringType  IN ('Ρύθμιση FT','Παράταση δόσης') OR L.RestructuringTypeCode = 'BL' THEN 'Extension'
					WHEN L.RestructuringType LIKE 'A%' OR  L.RestructuringType LIKE 'B%' OR
				      L.RestructuringType LIKE 'C%' OR  L.RestructuringType LIKE 'D%'  
					                                                                                 THEN L.RestructuringType
					 WHEN L.RestructuringType = 'ST'                                            THEN 'D1 Small Ticket'
				 	WHEN L.RestructuringProductKey IN ('4015','4016','6078','6079','7363','7362')  THEN 'Δικαστικές Ρυθμίσεις N3869'
				 	ELSE 'Συμβατικά'                                  END AS Category
					,C.Bucket                                       AS OperationalBucket
					,U.IFRSGrossCarryingAmount                                AS IFRSGrossCarryingAmountPrvM
					, U.BucketCategoryWithMateriality                      AS RiskBucketWithMateriality
		  			, U.NPLWithMaterialityFlag                        AS NPLIndicatorWithMateriality
		 		 	, U.DeNuncementFlag                                 AS DenouncementIndicator
					, CASE WHEN U.FPLUnderProbationFlag = 1 AND U.EBAStatus = 'FPL' THEN 'CFPL' ELSE U.EBAStatus END AS EBAStatus
					, C.SPV                                              AS SPV_DESC
		  			,	U.NonCoreFlag									   AS NonCoreFlag	
from LPFX364 L
LEFT JOIN `{env_prefix}-cdh`.reference.exchangerates  E ON E.BusinessDate = {run_date!r}  AND E.OriginalCurrency = 'CHF'
LEFT JOIN LPFXB302                       H1 ON L.OldExposureKey = H1.ExposureKey 
LEFT JOIN LPFXB302                       H2 ON L.NewExposureKey = H2.RelatedAccountNumber
LEFT JOIN extunionfile_temp      U ON L.OldExposureKey = U.ExposureNumber 
LEFT JOIN Qualco                          C ON L.OldExposureKey = C.ExposureKey 
where L.RNK=1 
'''
)
Modifications1.createOrReplaceTempView("Modifications1")

##### Modifications2

In [0]:
Modifications2 = spark.sql(
f'''
Select 
        L.*
		    , H1.AmountToWrittenOff  AS H1
			  , H2.AmountToWrittenOff  AS H2
			  , H3.AmountToWrittenOff	 AS H3
			  , RANK()   OVER (PARTITION BY NewExposureKey ORDER BY OldExposureKey) AS RNK_New
			  , COALESCE(H1.AmountToWrittenOff,COALESCE(H2.AmountToWrittenOff,H3.AmountToWrittenOff)) AS HaircutAmount 



from Modifications1 L
left join LPFXB302 H1 ON L.OldExposureKey= H1.ExposureKey 
LEFT JOIN LPFXB302 H2 ON L.IntermediateAccounts = H2.ExposureKey 
LEFT JOIN LPFXB302 H3 ON L.NewExposureKey = H3.RelatedAccountNumber
'''
)
Modifications2.createOrReplaceTempView("Modifications2")

##### Modifications3

In [0]:
Modifications3 = spark.sql(
f'''
SELECT 
		      H1
			, H2
			, H3
			, HaircutAmount
			, RNK_New
		    , AssetSegment
		    , RestructuringDate							AS ModificationDate
		    , PartyTaxRegistrationNumber 
		    , CDI
		    , AgreementKey            
		    , OldexposureKey
		    , NewExposureKey
		    , CAST(RestruturingAmount As Decimal(19,2)) AS ModificationAmount
		    , CAST(BalancePrvM As Decimal(19,2)) AS BalancePrvM
		    , IntermediateAccounts
		    , RestructuringProductKey    	AS ModificationProductCode
		    , Category
			,  CASE WHEN Category LIKE 'A%' THEN 'A'  
			        WHEN Category LIKE 'B%' THEN 'B'
			 	    WHEN Category LIKE 'C%' THEN 'C'
			 	    WHEN Category LIKE 'D%' THEN 'D'
			 	    ELSE Category   END AS Categoryfinal
		    , OperationalBucket
		    , CAST(Replace(IFRSGrossCarryingAmountPrvM,',','') AS FLOAT) AS IFRSGrossCarryingAmountPrvM
		    , RiskBucketWithMateriality
		    , COALESCE(NPLIndicatorWithMateriality,0) AS NPLIndicatorWithMateriality
		    , COALESCE(DenouncementIndicator,0)       AS DenouncementIndicator
		    , EBAStatus
			, SPV_DESC
		    , NonCoreFlag
		FROM Modifications2
'''
)
Modifications3.createOrReplaceTempView("Modifications3")

##### Modifications4

In [0]:
Modifications4_updated = spark.sql(
f'''
SELECT 
	     H1
	   , H2
	   , H3
	   , RANK()   OVER (PARTITION BY IntermediateAccounts,HaircutAmount  ORDER BY OldExposureKey) AS RNK_Endiamesoi
	   , RNK_New
	   , Case when (RANK()   OVER (PARTITION BY IntermediateAccounts,HaircutAmount  ORDER BY OldExposureKey) >1) then Null else HaircutAmount END AS HaircutAmount
	   , AssetSegment
	   , ModificationDate
	   , PartyTaxRegistrationNumber
	   , CDI
	   , AgreementKey
	   , OldExposureKey
	   , NewExposureKey
	   , ModificationAmount 
	   , BalancePrvM
	   , IntermediateAccounts
	   , ModificationProductCode
	   , Category
	   , Categoryfinal
	   , OperationalBucket
	   , IFRSGrossCarryingAmountPrvM
	   , RiskBucketWithMateriality
	   , NPLIndicatorWithMateriality
	   , DenouncementIndicator
	   , EBAStatus
	   , SPV_DESC
	   , NonCoreFlag
		FROM Modifications3 
  

'''
)
Modifications4_updated.createOrReplaceTempView("Modifications4_updated")

##### ModificationsFinal

In [0]:
ModificationsFinal = spark.sql(
f'''
SELECT 
		      AssetSegment
      		, ModificationDate
			, PartyTaxRegistrationNumber
			, CDI
			, AgreementKey
			, OldExposureKey
			, NewExposureKey
			, ModificationAmount
			, HaircutAmount
			, BalancePrvM
			, IntermediateAccounts
			, ModificationProductCode
			, Category
			, Categoryfinal
			, OperationalBucket
			, IFRSGrossCarryingAmountPrvM
			, RiskBucketWithMateriality
			, COALESCE(NPLIndicatorWithMateriality, 0) AS NPLIndicatorWithMateriality
			, COALESCE(DenouncementIndicator,0)        AS DenouncementIndicator
			, EBAStatus
			, SPV_DESC
		    , NonCoreFlag
	        , CONCAT(SUBSTRING(CAST(ModificationDate AS varchar(20)),1,4),SUBSTRING(CAST(ModificationDate AS varchar(20)),6,2))  AS Month
		    , CASE WHEN Category LIKE '%AC'     THEN LEFT(Category,2)
		           WHEN Category LIKE '%AC SPL' THEN CONCAT(LEFT(Category,2),' SPL')
		           WHEN Category LIKE '%C6%'    THEN 'C6'
				   WHEN (AssetSegment = 'ML' AND Category = 'Δικαστικές Ρυθμίσεις N3869') THEN 'ML Ν3869 (Δικ, Αποφάσεις)'
				   WHEN (AssetSegment = 'CL' AND Category = 'Δικαστικές Ρυθμίσεις N3869') THEN 'CL Ν3869 (Δικ, Αποφάσεις)'
			       WHEN (AssetSegment = 'SB' AND Category = 'Δικαστικές Ρυθμίσεις N3869') THEN 'SB Ν3869 (Δικ, Αποφάσεις)'
				   WHEN Category  = 'C4-greater discount' OR Category = 'C4 SU-greater discount' THEN 'C4'
				   WHEN (Category = 'C4 SPL-greater discount' OR Category = 'C4 SU SPL-greater discount') THEN 'C4 SPL'
				   WHEN Category  = 'B1 OCW 4738'                                 THEN 'B1'
				   WHEN Category  = 'C4 OCW 4738'                                 THEN 'C4'
				   ELSE Category                                                  END AS Category2	
		    , CASE WHEN (Categoryfinal = 'A' OR Categoryfinal = 'B' OR Categoryfinal = 'C' OR
		                Categoryfinal = 'D')                                     THEN 'Προιόντα Anchor'
				   ELSE 'Υφιστάμενα Προιόντα'                                       END AS Categoryfinal2
		    , CASE WHEN EBAStatus = 'PL'   THEN  1
		           WHEN EBAStatus = 'FPL'  THEN  2
		           WHEN EBAStatus = 'FNPL' THEN  3
		           WHEN EBAStatus = 'NPL'  THEN  4
		           ELSE 1                                                           END AS EBAStatusOldExposureFinal  
             ,cast(NULL as string) as RestructureType 
		   , CASE WHEN Categoryfinal = 'Δικαστικές Ρυθμίσεις N3869' AND HaircutAmount IS NOT NULL THEN 1 ELSE 0 END AS PosCourtDecPosHaircutFlag 
		FROM Modifications4_updated
'''
)
ModificationsFinal.createOrReplaceTempView("ModificationsFinal")

##### CoreModifications

In [0]:
CoreModifications = spark.sql(
f'''
SELECT 		
        AssetSegment
      , ModificationDate
      , PartyTaxRegistrationNumber
      , CDI
      , AgreementKey
      , OldExposureKey
      , NewExposureKey
      , ModificationAmount
      , cast(replace(HaircutAmount,',','.') as decimal(19,2)) as HaircutAmount
      , BalancePrvM
      , IntermediateAccounts
      , ModificationProductCode
      , Category
      , Categoryfinal 
      , OperationalBucket
      , IFRSGrossCarryingAmountPrvM
      , RiskBucketWithMateriality
      , NPLIndicatorWithMateriality
      , DenouncementIndicator
      , EBAStatus
      , SPV_DESC
      , Month
      , Category2 
      , Categoryfinal2 
      , EBAStatusOldExposureFinal  
      , RestructureType
      , PosCourtDecPosHaircutFlag
      , NonCoreFlag
    FROM ModificationsFinal 
'''
)
CoreModifications.createOrReplaceTempView("CoreModifications")

##### CountofNewAccountover1

In [0]:
CountofNewAccountover1 = spark.sql(
f'''
SELECT NewExposureKey
       ,ModificationAmount
       ,SUM(cast(HaircutAmount as decimal(19,2))) AS SumOfHaircutAmount
       FROM CoreModifications 
       WHERE MONTH(ModificationDate) = MONTH({run_date!r})
       GROUP BY NewExposureKey, ModificationAmount--, ModificationDate
       HAVING COUNT(NewExposureKey) > 1
         --  AND MONTH(ModificationDate) = MONTH({run_date!r})
'''
)
CountofNewAccountover1.createOrReplaceTempView("CountofNewAccountover1")

##### CountofOldAccountover1

In [0]:
CountofOldAccountover1 = spark.sql(
f'''
SELECT  OldExposureKey
      , SUM(ModificationAmount) AS SumOfModificationAmount
      , SUM(HaircutAmount) AS SumOfHaircutAmount
       FROM CoreModifications
       WHERE MONTH(ModificationDate) = MONTH({run_date!r}) 
       GROUP BY OldExposureKey--, ModificationDate
       HAVING COUNT(OldExposureKey) > 1
      --     AND MONTH(ModificationDate) = MONTH({run_date!r})
'''
)
CountofOldAccountover1.createOrReplaceTempView("CountofOldAccountover1")

##### CoreModifications_updated

In [0]:
CoreModifications_updated = spark.sql(
f'''
select  M.AssetSegment
      , M.ModificationDate
      , M.PartyTaxRegistrationNumber
      , M.CDI
      , M.AgreementKey
      , M.OldExposureKey
      , M.NewExposureKey
      , M.ModificationAmount
      , M.HaircutAmount
      , M.BalancePrvM
      , M.IntermediateAccounts
      , M.ModificationProductCode
      , M.Category
      , M.Categoryfinal 
      , M.OperationalBucket
      , M.IFRSGrossCarryingAmountPrvM
      , M.RiskBucketWithMateriality
      , M.NPLIndicatorWithMateriality
      , M.DenouncementIndicator
      , M.EBAStatus
      , M.SPV_DESC
      , M.Month
      , M.Category2 
      , M.Categoryfinal2 
      , M.EBAStatusOldExposureFinal  
      , M.RestructureType
      , M.PosCourtDecPosHaircutFlag
      , M.NonCoreFlag
  ,CASE WHEN (N.NewExposureKey IS  NULL AND O.OldExposureKey IS NULL AND MONTH(M.ModificationDate) = MONTH({run_date!r})) THEN M.ModificationAmount + M.HaircutAmount 
  WHEN (N.NewExposureKey IS NOT  NULL AND O.OldExposureKey IS NULL AND MONTH(M.ModificationDate) = MONTH({run_date!r})) THEN N.ModificationAmount + N.SumOfHaircutAmount
  when (N.NewExposureKey IS  NULL AND O.OldExposureKey IS NOT NULL AND MONTH(M.ModificationDate) = MONTH({run_date!r})) THEN O.SumOfModificationAmount + O.SumOfHaircutAmount END AS TotalBalanceBeforeRestructuring_OnBalance
  ,RANK()   OVER (PARTITION BY M.NewExposureKey ORDER BY M.HaircutAmount DESC , M.OldExposureKey ) AS RNK
 from CoreModifications M
 LEFT JOIN CountofNewAccountover1 N ON N.NewExposureKey = M.NewExposureKey
 LEFT JOIN CountofOldAccountover1 O ON O.OldExposureKey = M.OldExposureKey
'''
)
CoreModifications_updated.createOrReplaceTempView("CoreModifications_updated")

##### CoreModifications_final

In [0]:
CoreModifications_final = spark.sql(
f''' 
select  {run_date!r} AS BusinessDate
      , M.AssetSegment
      , M.ModificationDate
      , M.PartyTaxRegistrationNumber
      , M.CDI
      , M.AgreementKey
      , M.OldExposureKey
      , M.NewExposureKey
      , M.ModificationAmount
      , M.HaircutAmount
      , M.BalancePrvM
      , M.IntermediateAccounts
      , M.ModificationProductCode
      , M.Category
      , M.Categoryfinal 
      , M.OperationalBucket
      , M.IFRSGrossCarryingAmountPrvM
      , M.RiskBucketWithMateriality
      , M.NPLIndicatorWithMateriality
      , M.DenouncementIndicator
      , M.EBAStatus
      , M.SPV_DESC
      , M.Month
      , M.Category2 
      , M.Categoryfinal2 
      , M.EBAStatusOldExposureFinal  
      , M.RestructureType
      , M.PosCourtDecPosHaircutFlag
      , M.NonCoreFlag
      ,case when (N.NewExposureKey IS NOT NULL AND RNK > 1) then NULL else M.ModificationAmount end AS ModificationAmountHaircut
      ,case when (N.NewExposureKey IS NOT NULL AND RNK > 1) then NULL else M.TotalBalanceBeforeRestructuring_OnBalance end as TotalBalanceBeforeRestructuring_OnBalance
 FROM CoreModifications_updated         M
LEFT JOIN CountofNewAccountover1 N ON N.NewExposureKey = M.NewExposureKey
'''
)
CoreModifications_final.createOrReplaceTempView("CoreModifications_final")

##### final_df

In [0]:
final_df = spark.sql(
f'''
SELECT 
  CAST(BusinessDate AS date) AS BusinessDate
, CAST(AssetSegment AS string) AS AssetSegment
, CAST(ModificationDate AS date) AS ModificationDate
, CAST(PartyTaxRegistrationNumber AS string) AS PartyTaxRegistrationNumber
, CAST(CDI AS string) AS CDI
, CAST(AgreementKey AS string) AS AgreementKey
, CAST(OldExposureKey AS string) AS OldExposureKey
, CAST(NewExposureKey AS string) AS NewExposureKey
, CAST(ModificationAmount AS decimal(19,2)) AS ModificationAmount
, CAST(HaircutAmount AS decimal(19,2)) AS HaircutAmount
, CAST(BalancePrvM AS decimal(19,2)) AS BalancePrvM
, CAST(IntermediateAccounts AS string) AS IntermediateAccounts
, CAST(ModificationProductCode AS string) AS ModificationProductCode
, CAST(Category AS string) AS Category
, CAST(Categoryfinal AS string) AS Categoryfinal
, CAST(OperationalBucket AS string) AS OperationalBucket
, CAST(IFRSGrossCarryingAmountPrvM AS decimal(19,2)) AS IFRSGrossCarryingAmountPrvM
, CAST(RiskBucketWithMateriality AS string) AS RiskBucketWithMateriality
, CAST(NPLIndicatorWithMateriality AS string) AS NPLIndicatorWithMateriality
, CAST(DenouncementIndicator AS string) AS DenouncementIndicator
, CAST(EBAStatus AS string) AS EBAStatus
, CAST(SPV_DESC AS string) AS SPV
, CAST(Month AS string) AS Month
, CAST(Category2 AS string) AS Category2
, CAST(Categoryfinal2 AS string) AS Categoryfinal2
, CAST(EBAStatusOldExposureFinal AS int) AS EBAStatusOldExposureFinal
, CAST(RestructureType AS string) AS RestructureType
, CAST(PosCourtDecPosHaircutFlag AS int) AS PosCourtDecPosHaircutFlag
, CAST(NonCoreFlag AS string) AS NonCoreFlag
, CAST(ModificationAmountHaircut AS decimal(19,2)) AS ModificationAmountHaircut
, CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp
FROM CoreModifications_final A
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_Modifications: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["NewExposureKey", "OldExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
Modifications = spark.sql(
f'''
select * 
from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
Modifications.createOrReplaceTempView("Modifications")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/Modifications/"
file_name = "Modifications_" +run_date+ ".csv"
Modifications.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)

In [0]:
ModificationswithHaircut = spark.sql(
f'''
SELECT 	  AssetSegment AS BU
    , ModificationDate
	, Month
	, AgreementKey AS ContractNumber
    , CDI
    , OldExposureKey AS OldAccountNumber
	, NewExposureKey AS NewAccountNumber
	, BalancePrvM AS Balance_PrvM
	, ModificationAmount AS ModificationAmount
    , HaircutAmount
	, ModificationProductCode
    , Category
FROM {delivery_table_path}
WHERE BusinessDate = {run_date!r}
 AND (Categoryfinal = 'D' OR Categoryfinal= 'C')
'''
)
ModificationswithHaircut.createOrReplaceTempView("ModificationswithHaircut")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/ModificationswithHaircut/"
file_name = "ModificationswithHaircut_" +run_date+ ".csv"
ModificationswithHaircut.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)

In [0]:
ModificationsL3869 = spark.sql(
f'''
SELECT OldExposureKey AS OldAccountNumber
	 , IntermediateAccounts AS EndiamesoiAccounts
	 , PartyTaxRegistrationNumber AS AFM
	 , AgreementKey AS ContractNumber
	 , BalancePrvM AS Balance_PrvM
	 , NewExposureKey AS NewAccountNumber
	 , ModificationProductCode  AS  ModificationProductCode
	 , ModificationAmount AS ModificationAmount
	 , Month
     , AssetSegment AS BU
FROM {delivery_table_path}
WHERE BusinessDate = {run_date!r}
  AND Categoryfinal = 'Δικαστικές Ρυθμίσεις N3869'
'''
)
ModificationsL3869.createOrReplaceTempView("ModificationsL3869")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/ModificationsL3869/"
file_name = "ModificationsL3869_" +run_date+ ".csv"
ModificationsL3869.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)